# Two-Qubit Simulation of Hilbert-Space Inner-Product Change

This notebook implements quantum process tomography (QPT) for the two-qubit simulation
of a non-unitary inner-product (IP) change on a Rigetti superconducting processor.

The non-unitary action on qubit 26 is embedded into a larger two-qubit unitary involving
an ancilla qubit 27. Process tomography is performed on qubit 26 by post-selecting on
qubit 27 remaining in the ground state. The reconstructed Choi matrices are saved to
`data/raw_tomography/qubit_choi_matrices/`, and the raw tomography expectation values
are saved to `data/raw_tomography/qubit_tomography/`.

The 40 metric operators are parameterised by $(r, \\theta, \\phi)$ as defined in the
manuscript. Each corresponds to a 2x2 positive-definite matrix with largest eigenvalue 1.

In [ ]:
import sys
import csv
import time
from math import pi

import numpy as np
import scipy as sp
from scipy import linalg as la
import matplotlib.pyplot as plt

from tqdm import tqdm

import pyquil as py
from pyquil import get_qc, Program
from pyquil.api import QuantumComputer, WavefunctionSimulator
from pyquil.gates import *
from pyquil.paulis import *

from forest.benchmarking.compilation import basic_compile
from forest.benchmarking.tomography import *
from forest.benchmarking.tomography import linear_inv_state_estimate
from forest.benchmarking.tomography import _pauli_process_tomo_settings
from forest.benchmarking.observable_estimation import *
from forest.benchmarking.operator_tools import superoperator_transformations as supt
from forest.benchmarking.operator_tools import validate_superoperator as v

if sys.version_info < (3, 7):
    from pyquil.external.dataclasses import dataclass
else:
    from dataclasses import dataclass

%matplotlib inline

## Hardware initialisation

In [ ]:
qc = get_qc("Aspen-11")

# System qubit (26) and ancilla qubit (27)
qubits = [26, 27]
num_shots = 10000

## Metric operator parameter grid

Generates a grid of 40 points in $(r, \\theta, \\phi, \\alpha)$ space. Each point
parameterises a 2x2 positive-definite metric operator $\\eta$ with largest eigenvalue 1,
as defined in the manuscript (Eq. X). The parameter $\\alpha$ is derived from $r$ and
controls the mixing angle of the two-qubit embedding unitary.

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Circuit and tomography functions

In [ ]:
def get_process(qubits, vect):
    """Construct the two-qubit circuit that simulates the IP-change on qubits[0].

    The non-unitary operation on the system qubit (qubits[0]) is embedded into a
    two-qubit unitary using qubits[1] as an ancilla. The circuit consists of:
        RZ, RY rotations to align to the metric basis,
        an XY entangling gate parameterised by alpha,
        a CNOT to complete the embedding,
        and inverse rotations to return to the computational basis.

    Args:
        qubits: list of two qubit indices [system, ancilla].
        vect: parameter vector [r, theta, phi, alpha] for the metric operator.

    Returns:
        A pyQuil Program implementing the two-qubit embedding circuit.
    """
    process = Program()
    process += RZ(-1 * vect[2], qubits[0])
    process += RY(-1 * vect[1], qubits[0])
    process += XY(vect[3] - np.pi, qubits[0], qubits[1])
    process += CNOT(qubits[1], qubits[0])
    process += RY(vect[1], qubits[0])
    process += RZ(vect[2], qubits[0])
    return process

In [ ]:
def estimate_postselected_observables(
    qc: QuantumComputer,
    obs_expt: ObservablesExperiment,
    num_shots: int = 2000,
    active_reset: bool = False,
    use_basic_compile: bool = True,
) -> Iterable[ExperimentResult]:
    """Estimate Pauli expectation values for QPT on qubits[0], post-selecting on qubits[1]=|0>.

    This is a modified version of estimate_observables from
    forest.benchmarking.observable_estimation. The modification adds post-selection
    on the ancilla qubit (qubits[1]) being in the ground state, which implements
    the effective non-unitary channel on qubits[0]. Expectation values are
    renormalised by the post-selection probability.

    Args:
        qc: quantum computer object on which to run experiments.
        obs_expt: process tomography experiment with joint two-qubit observables.
        num_shots: number of shots per tomography setting.
        active_reset: whether to use active qubit reset between shots.
        use_basic_compile: whether to use basic_compile instead of the default compiler.

    Yields:
        ExperimentResult for each tomography setting, with expectation values
        renormalised by the post-selection probability.
    """
    if use_basic_compile:
        old_method = qc.compiler.quil_to_native_quil
        qc.compiler.quil_to_native_quil = basic_compile

    programs, meas_qubits = generate_experiment_programs(obs_expt, active_reset)
    for prog, meas_qs, settings in zip(programs, meas_qubits, obs_expt):

        results = qc.run_symmetrized_readout(prog, num_shots, symm_type=0, meas_qubits=meas_qs)

        # Post-select on ancilla qubit (index 1) in ground state |0>
        results_postselected = np.array([r for r in results if r[1] == [0]])

        for setting in settings:
            observable = setting.observable
            # Remove ancilla qubit from observable for single-qubit moment calculation
            observable = PauliTerm.from_compact_str(
                observable.compact_str()[:-len(f"Z{meas_qs[1]}")]
            )

            if len(results_postselected) == 0:
                yield ExperimentResult(
                    setting=setting,
                    expectation=0,
                    std_err=0,
                    total_counts=0,
                )
            else:
                obs_mean, obs_var = shots_to_obs_moments(
                    results_postselected, [meas_qs[0]], observable
                )
                # Renormalise expectation value by post-selection probability
                obs_mean = obs_mean * len(results_postselected) / num_shots
                yield ExperimentResult(
                    setting=setting,
                    expectation=obs_mean,
                    std_err=0,
                    total_counts=len(results_postselected),
                )

    if use_basic_compile:
        qc.compiler.quil_to_native_quil = old_method

In [ ]:
def get_postselected(qubits, results):
    """Strip the ancilla qubit from tomography experiment settings.

    After post-selection, the ancilla qubit label must be removed from each
    ExperimentSetting so that downstream analysis treats the results as
    single-qubit process tomography data.

    Args:
        qubits: list of two qubit indices [system, ancilla].
        results: list of ExperimentResult objects with joint two-qubit settings.

    Returns:
        list of ExperimentResult objects with ancilla qubit removed from settings.
    """
    postselected_results = []
    for r in results:
        single_str = r.setting.serializable()[:-len(f"Z{qubits[1]}")]
        sets = ExperimentSetting.from_str(single_str)
        postselected_results.append(
            ExperimentResult(
                setting=sets,
                expectation=r.expectation,
                std_err=r.std_err,
                total_counts=r.total_counts,
            )
        )
    return postselected_results

In [ ]:
def get_choi(qubits, postselected_results, num_shots):
    """Reconstruct the Choi matrix from post-selected QPT data via linear inversion.

    Constructs the measurement matrix from the input states and Pauli observables,
    augmented with identity rows to fix the trace of the Choi matrix. The system
    is solved via pseudoinverse.

    Args:
        qubits: list of two qubit indices [system, ancilla].
        postselected_results: list of ExperimentResult with ancilla qubit removed.
        num_shots: total number of shots used per tomography setting (used for
            renormalisation of identity rows).

    Returns:
        choi: (4, 4) complex numpy array, the reconstructed Choi matrix.
    """
    qs = [qubits[0]]

    # Extract input state density matrices for each preparation setting
    # Results are ordered: X+, X-, Y+, Y-, Z+, Z- (3 settings x 3 observables each)
    input_states = [
        state2matrix(postselected_results[i * 3].setting.in_state, qs)
        for i in range(6)
    ]

    # Build measurement matrix from Pauli observables
    measurement_matrix = np.vstack([
        vec(np.kron(
            state2matrix(result.setting.in_state, qs).conj(),
            pauli2matrix(result.setting.observable, qs)
        )).conj().T
        for result in postselected_results
    ])

    # Augment with identity rows to fix the trace (one per input state)
    identity_rows = np.vstack([
        vec(np.kron(rho.conj(), np.eye(2))).conj().T
        for rho in input_states
    ])
    measurement_matrix = np.vstack((measurement_matrix, identity_rows))

    # Build expectation value vector, augmented with post-selection probabilities
    expectations = np.array([result.expectation for result in postselected_results])
    trace_values = np.array([
        postselected_results[i * 3].total_counts / num_shots
        for i in range(6)
    ])
    expectations = np.hstack((expectations, trace_values))

    # Solve via pseudoinverse and reshape to Choi matrix
    rho = np.linalg.pinv(measurement_matrix) @ expectations
    choi = unvec(rho)
    return choi

## Run process tomography for all 40 metric operators

In [ ]:
import os
import csv
import numpy as np

# Output directories (relative to repository root)
TOMOGRAPHY_DIR = "data/raw_tomography/qubit_tomography"
CHOI_DIR = "data/raw_tomography/qubit_choi_matrices"
os.makedirs(TOMOGRAPHY_DIR, exist_ok=True)
os.makedirs(CHOI_DIR, exist_ok=True)

grid = eta_grid()
processes = [get_process(qubits, vect) for vect in grid]

for k, process in tqdm(enumerate(processes)):
    # Set up QPT on system qubit only, measuring ancilla in Z basis
    tomo_expt_settings = list(_pauli_process_tomo_settings(qubits=[qubits[0]]))
    added_tomo_str = [
        t.serializable() + f"*Z{qubits[1]}" for t in tomo_expt_settings
    ]
    added_tomo_expt_settings = [
        ExperimentSetting.from_str(s) for s in added_tomo_str
    ]
    tomography_experiment = ObservablesExperiment(
        settings=added_tomo_expt_settings, program=process
    )

    # Collect raw tomography data with post-selection on ancilla
    results = list(
        estimate_postselected_observables(qc, tomography_experiment, num_shots)
    )

    # Save raw tomography expectation values
    tomo_path = os.path.join(TOMOGRAPHY_DIR, f"qubit_tomography_metric_{k+1:02d}.csv")
    with open(tomo_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["in_state", "observable", "expectation", "std_err", "total_counts"])
        for r in results:
            writer.writerow([
                r.setting.in_state,
                r.setting.observable,
                r.expectation,
                r.std_err,
                r.total_counts,
            ])

    # Remove ancilla qubit label from settings for Choi reconstruction
    postselected_results = get_postselected(qubits, results)

    # Reconstruct and save Choi matrix
    choi = get_choi(qubits, postselected_results, num_shots)
    choi_path = os.path.join(CHOI_DIR, f"qubit_choi_metric_{k+1:02d}.npy")
    np.save(choi_path, choi)

print("Done. Tomography data saved to", TOMOGRAPHY_DIR)
print("Choi matrices saved to", CHOI_DIR)